# Chapter 7 &mdash; The Formal NFA: $\delta$ Returns a Set

**Concept 4 of the Chapter 7 decomposition:** *The Formal NFA: $(Q,\Sigma,\delta,Q_0,F)$ with $\delta: Q\times\Sigma_\varepsilon\to{\cal P}(Q)$*

Three changes from the DFA tuple: $\delta$ returns a <i>set</i>, accepts $\varepsilon$, and the start is a set $Q_0$.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter7/Concept-Formal-NFA-Tuple/Concept-Formal-NFA-Tuple.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


An NFA is $(Q,\Sigma,\delta,Q_0,F)$ with exactly three differences from a DFA:

* $\delta: Q\times\Sigma_\varepsilon \to \mathcal{P}(Q)$ &mdash; the result is a **set of
  states**, possibly empty;
* the input alphabet is $\Sigma_\varepsilon = \Sigma\cup\{\varepsilon\}$;
* the start is a **set** $Q_0 \subseteq Q$, not a single state.

Returning $\emptyset$ is how a token **dies**, so an NFA's $\delta$ is **total** in a
trivial sense &mdash; there is no need to totalize, and no black hole.

## 2. Definitions

### The five keys, NFA flavour

In [ ]:
N = md2mc('''NFA
I : 0 | 1 -> I
I : 1 -> A
A : 0 | 1 -> F
''')
for k in ['Q', 'Sigma', 'Delta', 'Q0', 'F']:
    v = N[k]
    print("%-6s : %s" % (k, sorted(v) if isinstance(v, set) else v))

### $\delta$ as a table of **sets**

In [ ]:
def delta_table(N):
    for q in sorted(N["Q"]):
        for a in sorted(N["Sigma"]) + ['']:
            r = step_nfa(N, q, a)
            print("   delta(%-3s, %-3s) = %s" % (q, repr(a), sorted(r) if r else "{}  <- token dies"))

## 3. Tests

Every result is a **set** &mdash; sometimes empty, sometimes several states.

In [ ]:
delta_table(N)
assert all(isinstance(step_nfa(N, q, a), set)
           for q in N["Q"] for a in list(N["Sigma"]) + [''])

`Q0` is a set, and can hold more than one state.

In [ ]:
Multi = md2mc('''NFA
I1 : 0 -> F
I2 : 1 -> F
''')
print("Q0 =", sorted(Multi["Q0"]), " <- two initial states, legal for an NFA")
assert len(Multi["Q0"]) == 2
print("accepts '0'?", accepts_nfa(Multi, '0'), "  accepts '1'?", accepts_nfa(Multi, '1'))
assert accepts_nfa(Multi, '0') and accepts_nfa(Multi, '1')
print("\nThe same markdown as a DFA would be rejected: 'DFA with 2 starting states'.")

No totalization is needed: $\emptyset$ already means 'reject along this path'.

In [ ]:
print("delta(A, '0') from the first machine :", sorted(step_nfa(N, 'A', '0')))
print("delta(F, '0')                        :", step_nfa(N, 'F', '0'), " <- empty, no black hole needed")
assert step_nfa(N, 'F', '0') == set()

Building one with `mk_nfa` gives the same machine.

In [ ]:
N2 = mk_nfa(N["Q"], N["Sigma"], N["Delta"], N["Q0"], N["F"])
from itertools import product
assert all(accepts_nfa(N, ''.join(p)) == accepts_nfa(N2, ''.join(p))
           for k in range(8) for p in product('01', repeat=k))
print("mk_nfa reconstruction agrees on all strings up to length 7")

## 4. Exercises


1. Which of the three differences is the one that really creates nondeterminism?
2. Why does an NFA need no black-hole state?
3. Write a DFA's five-tuple as an NFA's five-tuple. What changes?

In [ ]:
# Your work for the exercises above.